In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import brainmass
import brainstate
import brainunit as u
import jax
import jax.numpy as jnp
import numpy as np
brainstate.environ.set(dt=0.1 * u.ms)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


# Testing

Testing guidelines for `brainmass`.

## Running Tests

Run all tests:

```bash
pytest tests/
```

Run specific test file:

```bash
pytest tests/test_models.py
```

Run with coverage:

```bash
pytest --cov=brainmass tests/
```

## Test Structure

```text
tests/
├── test_models.py          # Neural mass model tests
├── test_noise.py           # Noise process tests
├── test_coupling.py        # Coupling mechanism tests
├── test_forward.py         # Forward model tests
└── test_utils.py           # Utility function tests
```

## Writing Tests

### Basic Test

In [2]:
import pytest
import jax.numpy as jnp
import brainmass

def test_hopf_oscillator():
    # Create model
    model = brainmass.HopfStep(in_size=10, w=0.2)

    # Initialize
    model.init_all_states()

    # Test update
    output = model.update()

    # Assertions
    assert output.shape == (10,)
    assert not jnp.any(jnp.isnan(output))

### Parametrized Tests

In [3]:
@pytest.mark.parametrize("in_size", [1, 10, 100])
def test_model_shapes(in_size):
    model = brainmass.WilsonCowanStep(in_size=in_size)
    model.init_all_states()

    output = model.update(rE_inp=0.5, rI_inp=0.2)
    assert output.shape == (in_size,)

### Batch Testing

In [4]:
@pytest.mark.parametrize("batch_size", [None, 1, 32])
def test_batching(batch_size):
    model = brainmass.HopfStep(in_size=5, w=0.2)
    model.init_all_states(batch_size=batch_size)

    output = model.update()

    if batch_size is None:
        assert output.shape == (5,)
    else:
        assert output.shape == (batch_size, 5)

## Test Best Practices

1. **Test one thing per test**
2. **Use descriptive test names**
3. **Check shapes, values, and edge cases**
4. **Test with and without batch dimension**
5. **Test with units where applicable**

## Coverage

Aim for >80% code coverage:

```bash
pytest --cov=brainmass --cov-report=html tests/
# Open htmlcov/index.html
```

## Continuous Integration

Tests run automatically on:

- Every pull request
- Pushes to main branch
- Nightly builds

Ensure tests pass before submitting PR.

## See Also

- pytest documentation
- {doc}`contributing` - Contribution process